**生成式 AI 使用说明**：本作业中使用生成式 AI 工具时，适用与协作相同的课程政策。和其他协作者一样，每位学生都必须独立写出自己的解答，不能直接依赖交互输出；提交内容中还应注明协作的性质。使用生成式 AI 工具实质性完成作业内容不符合本作业的精神，也会违反 [Honor Code](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)。

**学生声明（必须填写）**

**SUNet id：**  
_填写你的 SUNet id_

**你是否使用了生成式 AI 工具（例如 ChatGPT、Copilot 等）？**  
- [ ] 否  
- [ ] 是（请在下方说明）

**如果使用了，请说明你如何使用生成式 AI，以及用于作业的哪些部分：**  
_填写你的回答_

**请确认所有提交内容都反映你自己的理解，并且你没有依赖 AI 完成作业的实质性部分：**  
- [ ] 我确认

In [ ]:
# 将你的 Google Drive 挂载到 Colab VM。
from google.colab import drive
drive.mount('/content/drive')

# TODO：填写 Drive 中保存解压后
# 作业文件夹，例如 'cs231n/assignments/assignment2/'
FOLDERNAME = 'cs231n/assignments/assignment2/'
assert FOLDERNAME is not None, "[!] Enter the foldername."

# 现在已经挂载 Drive，下面确保
# Colab VM 的 Python 解释器可以加载
# 其中的 Python 文件。
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# 将 CIFAR-10 数据集下载到你的 Drive
# 如果它还不存在。
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
!bash get_datasets.sh
%cd /content/drive/My\ Drive/$FOLDERNAME

# Batch Normalization
让深度网络更容易训练的一种方法，是使用更复杂的优化过程，例如 SGD+momentum、RMSProp 或 Adam。另一种方法是改变网络结构，让它本身更易于训练。Batch normalization 就是这类思路中的一种方法，由文献 [1] 于 2015 年提出。

要理解 batch normalization 的目标，首先要认识到：机器学习方法通常在输入数据具有零均值、单位方差且特征之间相关性较低时表现更好。训练神经网络时，我们可以在把数据送入网络之前先做预处理，显式降低输入特征之间的相关性。这样能保证网络第一层看到的数据分布更友好。然而，即使输入数据已经预处理，网络深层的激活值也很可能不再满足低相关、零均值和单位方差，因为它们是前面层的输出。更糟的是，在训练过程中，每一层的权重不断更新，各层特征分布也会随之漂移。

文献 [1] 的作者假设，深度神经网络内部特征分布的漂移会让训练更加困难。为解决这个问题，他们提出在网络中插入对 batch 做归一化的层。训练时，这类层使用一个 minibatch 的数据估计每个特征的均值和标准差，并用这些估计值对 minibatch 的特征做中心化和归一化。训练过程中还会维护这些均值和标准差的滑动平均；测试时使用这些滑动平均来中心化和归一化特征。

这种归一化策略有可能降低网络的表达能力，因为某些层在最优情况下可能确实需要非零均值或非单位方差的特征。因此，batch normalization 层还为每个特征维度加入可学习的平移参数和缩放参数。

[1] [Sergey Ioffe and Christian Szegedy, "Batch Normalization: Accelerating Deep Network Training by Reducing
Internal Covariate Shift", ICML 2015.](https://arxiv.org/abs/1502.03167)

In [ ]:
# 设置单元。
import time
import numpy as np
import matplotlib.pyplot as plt
from cs231n.data_utils import get_CIFAR10_data
from cs231n.gradient_check import eval_numerical_gradient, eval_numerical_gradient_array
from cs231n.solver import Solver

%matplotlib inline
plt.rcParams["figure.figsize"] = (10.0, 8.0)  # 设置图像的默认大小。
plt.rcParams["image.interpolation"] = "nearest"
plt.rcParams["image.cmap"] = "gray"

import sys
import types
import importlib

if "imp" not in sys.modules:
    imp = types.ModuleType("imp")
    imp.reload = importlib.reload
    sys.modules["imp"] = imp

%load_ext autoreload
%autoreload 2

def rel_error(x, y):
    """Returns relative error."""
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

def print_mean_std(x,axis=0):
    print(f"  means: {x.mean(axis=axis)}")
    print(f"  stds:  {x.std(axis=axis)}\n")

In [ ]:
# 加载预处理后的 CIFAR-10 数据。
data = get_CIFAR10_data()
for k, v in list(data.items()):
    print(f"{k}: {v.shape}")

# Batch Normalization：前向传播
在 `cs231n/layers.py` 文件中，在 `batchnorm_forward` 函数里实现 batch normalization 的前向传播。完成后运行下面的代码测试你的实现。

上面 [1] 中链接的论文可能会有帮助。

In [ ]:
from cs231n.layers import *

# Check 训练时 前向传播 by checking 均值 并 方差
# of 特征 both before 并 after batch 归一化

# Simulate 前向传播 用于 a two-层 network.
np.random.seed(231)
N, D1, D2, D3 = 200, 50, 60, 3
X = np.random.randn(N, D1)
W1 = np.random.randn(D1, D2)
W2 = np.random.randn(D2, D3)
a = np.maximum(0, X.dot(W1)).dot(W2)

print('Before batch normalization:')
print_mean_std(a,axis=0)

gamma = np.ones((D3,))
beta = np.zeros((D3,))

# Means 应为 接近 zero 并 标准差 接近 one.
print('After batch normalization (gamma=1, beta=0)')
a_norm, _ = batchnorm_forward(a, gamma, beta, {'mode': 'train'})
print_mean_std(a_norm,axis=0)

gamma = np.asarray([1.0, 2.0, 3.0])
beta = np.asarray([11.0, 12.0, 13.0])

# Now 均值 应为 接近 beta 并 标准差 接近 gamma.
print('After batch normalization (gamma=', gamma, ', beta=', beta, ')')
a_norm, _ = batchnorm_forward(a, gamma, beta, {'mode': 'train'})
print_mean_std(a_norm,axis=0)

In [ ]:
# Check 测试时 前向传播 by running 训练时
# 前向传播 many times 到 warm up running averages, 并 然后
# checking 均值 并 方差 的 activations after a 测试时
# 前向传播.

np.random.seed(231)
N, D1, D2, D3 = 200, 50, 60, 3
W1 = np.random.randn(D1, D2)
W2 = np.random.randn(D2, D3)

bn_param = {'mode': 'train'}
gamma = np.ones(D3)
beta = np.zeros(D3)

for t in range(50):
  X = np.random.randn(N, D1)
  a = np.maximum(0, X.dot(W1)).dot(W2)
  batchnorm_forward(a, gamma, beta, bn_param)

bn_param['mode'] = 'test'
X = np.random.randn(N, D1)
a = np.maximum(0, X.dot(W1)).dot(W2)
a_norm, _ = batchnorm_forward(a, gamma, beta, bn_param)

# Means 应为 接近 zero 并 标准差 接近 one, but 将 be
# noisier than 训练时 前向传播es.
print('After batch normalization (test-time):')
print_mean_std(a_norm,axis=0)

# Batch Normalization：反向传播
现在在 `batchnorm_backward` 函数中实现 batch normalization 的反向传播。

推导反向传播时，你应该写出 batch normalization 的计算图，并对每个中间节点做反向传播。有些中间量可能会流向多个分支；在反向传播中要确保把这些分支上的梯度相加。上面 [1] 中链接的论文可能会有帮助。

完成后，运行下面的代码对反向传播做数值检查。

_提示：[这个资源](https://www.adityaagrawal.net/blog/deep_learning/bprop_batch_norm) 解释了如何从论文推导梯度。_

In [ ]:
# Gradient check batchnorm 反向传播.
np.random.seed(231)
N, D = 4, 5
x = 5 * np.random.randn(N, D) + 12
gamma = np.random.randn(D)
beta = np.random.randn(D)
dout = np.random.randn(N, D)

bn_param = {'mode': 'train'}
fx = lambda x: batchnorm_forward(x, gamma, beta, bn_param)[0]
fg = lambda a: batchnorm_forward(x, a, beta, bn_param)[0]
fb = lambda b: batchnorm_forward(x, gamma, b, bn_param)[0]

dx_num = eval_numerical_gradient_array(fx, x, dout)
da_num = eval_numerical_gradient_array(fg, gamma.copy(), dout)
db_num = eval_numerical_gradient_array(fb, beta.copy(), dout)

_, cache = batchnorm_forward(x, gamma, beta, bn_param)
dx, dgamma, dbeta = batchnorm_backward(dout, cache)

# 你应该 expect 到 see 相对误差s between 1e-13 并 1e-8.
print('dx error: ', rel_error(dx_num, dx))
print('dgamma error: ', rel_error(da_num, dgamma))
print('dbeta error: ', rel_error(db_num, dbeta))

# Batch Normalization：另一种反向传播实现
课上我们讨论过 sigmoid 反向传播的两种实现方式。一种策略是写出由简单操作组成的计算图，并对所有中间值做反向传播。另一种策略是在纸上直接推导导数。例如，你可以通过在纸上化简梯度，得到 sigmoid 函数反向传播的一个非常简洁公式。

有些令人意外的是，batch normalization 的反向传播也可以做类似化简。

在前向传播中，给定一组输入 $X=\begin{bmatrix}x_1\\x_2\\...\\x_N\end{bmatrix}$，

我们首先计算均值 $\mu$ 和方差 $v$。有了 $\mu$ 和 $v$ 后，可以计算标准差 $\sigma$ 和归一化数据 $Y$。下面的公式和计算图说明了这个计算过程，其中 $y_i$ 是向量 $Y$ 的第 i 个元素。

\begin{align}
& \mu=\frac{1}{N}\sum_{k=1}^N x_k  &  v=\frac{1}{N}\sum_{k=1}^N (x_k-\mu)^2 \\
& \sigma=\sqrt{v+\epsilon}         &  y_i=\frac{x_i-\mu}{\sigma}
\end{align}

<img src="https://raw.githubusercontent.com/cs231n/cs231n.github.io/master/assets/a2/batchnorm_graph.png">

反向传播中的核心问题是：给定我们收到的上游梯度 $\frac{\partial L}{\partial Y}$，计算 $\frac{\partial L}{\partial X}$。根据微积分中的链式法则：
$\frac{\partial L}{\partial X} = \frac{\partial L}{\partial Y} \cdot \frac{\partial Y}{\partial X}$。

未知且困难的部分是 $\frac{\partial Y}{\partial X}$。我们可以先逐步推导局部梯度：
$\frac{\partial v}{\partial X}$、$\frac{\partial \mu}{\partial X}$、$\frac{\partial \sigma}{\partial v}$、$\frac{\partial Y}{\partial \sigma}$ 和 $\frac{\partial Y}{\partial \mu}$，然后用链式法则把这些梯度（它们会以向量形式出现）正确组合起来，计算 $\frac{\partial Y}{\partial X}$。

如果直接对需要矩阵乘法的 $X$ 和 $Y$ 做梯度推理比较困难，可以先从单个元素 $x_i$ 和 $y_i$ 出发：这时你需要先用链式法则计算中间量 $\frac{\partial \mu}{\partial x_i}$、$\frac{\partial v}{\partial x_i}$、$\frac{\partial \sigma}{\partial x_i}$，再把这些部分组合起来得到 $\frac{\partial y_i}{\partial x_i}$。

为了便于实现，请尽量把每个中间梯度推导都化简到最简形式。

完成推导后，在 `batchnorm_backward_alt` 函数中实现简化版 batch normalization 反向传播，并运行下面的代码比较两种实现。两种实现应该得到几乎相同的结果，但替代实现应该稍快一些。

_提示：https://cs.stanford.edu/people/jcjohns/batchnorm.pdf。注意最终推导中应包含 γ。_

_请注意，这个 pdf 中公式 (8) 应写为：_ 
$$\frac{1}{\sigma} \frac{\partial}{\partial x_i} (x_j - \mu)
- \frac{1}{\sigma^2} \frac{\partial \sigma}{\partial x_i} (x_j - \mu)$$

In [ ]:
np.random.seed(231)
N, D = 100, 500
x = 5 * np.random.randn(N, D) + 12
gamma = np.random.randn(D)
beta = np.random.randn(D)
dout = np.random.randn(N, D)

bn_param = {'mode': 'train'}
out, cache = batchnorm_forward(x, gamma, beta, bn_param)

t1 = time.time()
dx1, dgamma1, dbeta1 = batchnorm_backward(dout, cache)
t2 = time.time()
dx2, dgamma2, dbeta2 = batchnorm_backward_alt(dout, cache)
t3 = time.time()

print('dx difference: ', rel_error(dx1, dx2))
print('dgamma difference: ', rel_error(dgamma1, dgamma2))
print('dbeta difference: ', rel_error(dbeta1, dbeta2))
print('speedup: %.2fx' % ((t2 - t1) / (t3 - t2)))

# 带 Batch Normalization 的全连接网络
现在你已经有了可用的 batch normalization 实现，回到 `cs231n/classifiers/fc_net.py` 文件中的 `FullyConnectedNet`。回忆一下，你在 Assignment 1 中已经实现了网络初始化、前向传播和反向传播。把那份实现复制到这里，并修改它以加入 batch normalization。

具体来说，当构造函数中的 `normalization` 标志设置为 `"batchnorm"` 时，你应该在每个 ReLU 非线性之前插入一个 batch normalization 层。网络最后一层的输出不应该归一化。完成后，运行下面的代码对实现做梯度检查。

**提示：** 你可能会发现，定义一个类似 `cs231n/layer_utils.py` 中那些辅助层的额外 helper layer 会很方便。

In [ ]:
from cs231n.classifiers.fc_net import *
from cs231n.gradient_check import *

np.random.seed(231)
N, D, H1, H2, C = 2, 15, 20, 30, 10
X = np.random.randn(N, D)
y = np.random.randint(C, size=(N,))

# 你应该 expect 损失 between 1e-4~1e-10 用于 W,
# 损失 between 1e-08~1e-10 用于 b,
# and 损失 between 1e-08~1e-09 用于 beta 并 gammas.
for reg in [0, 3.14]:
  print('Running check with reg = ', reg)
  model = FullyConnectedNet([H1, H2], input_dim=D, num_classes=C,
                            reg=reg, weight_scale=5e-2, dtype=np.float64,
                            normalization='batchnorm')

  loss, grads = model.loss(X, y)
  print('Initial loss: ', loss)

  for name in sorted(grads):
    f = lambda _: model.loss(X, y)[0]
    grad_num = eval_numerical_gradient(f, model.params[name], verbose=False, h=1e-5)
    print('%s relative error: %.2e' % (name, rel_error(grad_num, grads[name])))
  if reg == 0: print()

# 深层网络中的 Batch Normalization
运行下面的代码，在 1000 个训练样本的子集上分别训练有 batch normalization 和没有 batch normalization 的六层网络。

In [ ]:
np.random.seed(231)

# Try 训练 a very deep net 使用 batchnorm.
hidden_dims = [100, 100, 100, 100, 100]

num_train = 1000
small_data = {
  'X_train': data['X_train'][:num_train],
  'y_train': data['y_train'][:num_train],
  'X_val': data['X_val'],
  'y_val': data['y_val'],
}

weight_scale = 2e-2
bn_model = FullyConnectedNet(hidden_dims, weight_scale=weight_scale, normalization='batchnorm')
model = FullyConnectedNet(hidden_dims, weight_scale=weight_scale, normalization=None)

print('Solver with batch norm:')
bn_solver = Solver(bn_model, small_data,
                num_epochs=10, batch_size=50,
                update_rule='adam',
                optim_config={
                  'learning_rate': 1e-3,
                },
                verbose=True,print_every=20)
bn_solver.train()

print('\nSolver without batch norm:')
solver = Solver(model, small_data,
                num_epochs=10, batch_size=50,
                update_rule='adam',
                optim_config={
                  'learning_rate': 1e-3,
                },
                verbose=True, print_every=20)
solver.train()

运行下面的代码可视化上面训练的两个网络的结果。你应该会发现，使用 batch normalization 可以帮助网络收敛得更快。

In [ ]:
def plot_training_history(title, label, baseline, bn_solvers, plot_fn, bl_marker='.', bn_marker='.', labels=None):
    """utility 函数 用于 plotting 训练 history"""
    plt.title(title)
    plt.xlabel(label)
    bn_plots = [plot_fn(bn_solver) for bn_solver in bn_solvers]
    bl_plot = plot_fn(baseline)
    num_bn = len(bn_plots)
    for i in range(num_bn):
        label='with_norm'
        if labels is not None:
            label += str(labels[i])
        plt.plot(bn_plots[i], bn_marker, label=label)
    label='baseline'
    if labels is not None:
        label += str(labels[0])
    plt.plot(bl_plot, bl_marker, label=label)
    plt.legend(loc='lower center', ncol=num_bn+1)


plt.subplot(3, 1, 1)
plot_training_history('Training loss','Iteration', solver, [bn_solver],\
                      lambda x: x.loss_history, bl_marker='o', bn_marker='o')
plt.subplot(3, 1, 2)
plot_training_history('Training accuracy','Epoch', solver, [bn_solver],\
                      lambda x: x.train_acc_history, bl_marker='-o', bn_marker='-o')
plt.subplot(3, 1, 3)
plot_training_history('Validation accuracy','Epoch', solver, [bn_solver],\
                      lambda x: x.val_acc_history, bl_marker='-o', bn_marker='-o')

plt.gcf().set_size_inches(15, 15)
plt.show()

# Batch Normalization 与初始化
现在我们运行一个小实验，研究 batch normalization 和权重初始化之间的相互作用。

第一个单元会使用不同的权重初始化尺度，分别训练有 batch normalization 和没有 batch normalization 的八层网络。第二个单元会绘制训练准确率、验证集准确率和训练损失随权重初始化尺度变化的曲线。

In [ ]:
np.random.seed(231)

# Try 训练 a very deep net 使用 batchnorm.
hidden_dims = [50, 50, 50, 50, 50, 50, 50]
num_train = 1000
small_data = {
  'X_train': data['X_train'][:num_train],
  'y_train': data['y_train'][:num_train],
  'X_val': data['X_val'],
  'y_val': data['y_val'],
}

bn_solvers_ws = {}
solvers_ws = {}
weight_scales = np.logspace(-4, 0, num=20)
for i, weight_scale in enumerate(weight_scales):
    print('Running weight scale %d / %d' % (i + 1, len(weight_scales)))
    bn_model = FullyConnectedNet(hidden_dims, weight_scale=weight_scale, normalization='batchnorm')
    model = FullyConnectedNet(hidden_dims, weight_scale=weight_scale, normalization=None)

    bn_solver = Solver(bn_model, small_data,
                  num_epochs=10, batch_size=50,
                  update_rule='adam',
                  optim_config={
                    'learning_rate': 1e-3,
                  },
                  verbose=False, print_every=200)
    bn_solver.train()
    bn_solvers_ws[weight_scale] = bn_solver

    solver = Solver(model, small_data,
                  num_epochs=10, batch_size=50,
                  update_rule='adam',
                  optim_config={
                    'learning_rate': 1e-3,
                  },
                  verbose=False, print_every=200)
    solver.train()
    solvers_ws[weight_scale] = solver

In [ ]:
# Plot 结果 的 weight 缩放 experiment.
best_train_accs, bn_best_train_accs = [], []
best_val_accs, bn_best_val_accs = [], []
final_train_loss, bn_final_train_loss = [], []

for ws in weight_scales:
  best_train_accs.append(max(solvers_ws[ws].train_acc_history))
  bn_best_train_accs.append(max(bn_solvers_ws[ws].train_acc_history))

  best_val_accs.append(max(solvers_ws[ws].val_acc_history))
  bn_best_val_accs.append(max(bn_solvers_ws[ws].val_acc_history))

  final_train_loss.append(np.mean(solvers_ws[ws].loss_history[-100:]))
  bn_final_train_loss.append(np.mean(bn_solvers_ws[ws].loss_history[-100:]))

plt.subplot(3, 1, 1)
plt.title('Best val accuracy vs. weight initialization scale')
plt.xlabel('Weight initialization scale')
plt.ylabel('Best val accuracy')
plt.semilogx(weight_scales, best_val_accs, '-o', label='baseline')
plt.semilogx(weight_scales, bn_best_val_accs, '-o', label='batchnorm')
plt.legend(ncol=2, loc='lower right')

plt.subplot(3, 1, 2)
plt.title('Best train accuracy vs. weight initialization scale')
plt.xlabel('Weight initialization scale')
plt.ylabel('Best training accuracy')
plt.semilogx(weight_scales, best_train_accs, '-o', label='baseline')
plt.semilogx(weight_scales, bn_best_train_accs, '-o', label='batchnorm')
plt.legend()

plt.subplot(3, 1, 3)
plt.title('Final training loss vs. weight initialization scale')
plt.xlabel('Weight initialization scale')
plt.ylabel('Final training loss')
plt.semilogx(weight_scales, final_train_loss, '-o', label='baseline')
plt.semilogx(weight_scales, bn_final_train_loss, '-o', label='batchnorm')
plt.legend()
plt.gca().set_ylim(1.0, 3.5)

plt.gcf().set_size_inches(15, 15)
plt.show()

## 内联问题 1：
描述这个实验的结果。权重初始化尺度对有 batch normalization 和没有 batch normalization 的模型分别有什么不同影响？为什么？

## 回答：
[在此填写]

# Batch Normalization 与 Batch Size
现在我们运行一个小实验，研究 batch normalization 和 batch size 之间的相互作用。

第一个单元会使用不同 batch size，分别训练有 batch normalization 和没有 batch normalization 的六层网络。第二个单元会绘制训练准确率和验证集准确率随时间变化的曲线。

In [ ]:
def run_batchsize_experiments(normalization_mode):
    np.random.seed(231)

    # Try 训练 a very deep net 使用 batchnorm.
    hidden_dims = [100, 100, 100, 100, 100]
    num_train = 1000
    small_data = {
      'X_train': data['X_train'][:num_train],
      'y_train': data['y_train'][:num_train],
      'X_val': data['X_val'],
      'y_val': data['y_val'],
    }
    n_epochs=10
    weight_scale = 2e-2
    batch_sizes = [5,10,50]
    lr = 10**(-3.5)
    solver_bsize = batch_sizes[0]

    print('No normalization: batch size = ',solver_bsize)
    model = FullyConnectedNet(hidden_dims, weight_scale=weight_scale, normalization=None)
    solver = Solver(model, small_data,
                    num_epochs=n_epochs, batch_size=solver_bsize,
                    update_rule='adam',
                    optim_config={
                      'learning_rate': lr,
                    },
                    verbose=False)
    solver.train()

    bn_solvers = []
    for i in range(len(batch_sizes)):
        b_size=batch_sizes[i]
        print('Normalization: batch size = ',b_size)
        bn_model = FullyConnectedNet(hidden_dims, weight_scale=weight_scale, normalization=normalization_mode)
        bn_solver = Solver(bn_model, small_data,
                        num_epochs=n_epochs, batch_size=b_size,
                        update_rule='adam',
                        optim_config={
                          'learning_rate': lr,
                        },
                        verbose=False)
        bn_solver.train()
        bn_solvers.append(bn_solver)

    return bn_solvers, solver, batch_sizes

batch_sizes = [5,10,50]
bn_solvers_bsize, solver_bsize, batch_sizes = run_batchsize_experiments('batchnorm')

In [ ]:
plt.subplot(2, 1, 1)
plot_training_history('Training accuracy (Batch Normalization)','Epoch', solver_bsize, bn_solvers_bsize,\
                      lambda x: x.train_acc_history, bl_marker='-^', bn_marker='-o', labels=batch_sizes)
plt.subplot(2, 1, 2)
plot_training_history('Validation accuracy (Batch Normalization)','Epoch', solver_bsize, bn_solvers_bsize,\
                      lambda x: x.val_acc_history, bl_marker='-^', bn_marker='-o', labels=batch_sizes)

plt.gcf().set_size_inches(15, 10)
plt.show()

## 内联问题 2：
描述这个实验的结果。这说明 batch normalization 和 batch size 之间有什么关系？为什么会观察到这种关系？

## 回答：
[在此填写]

# Layer Normalization
Batch normalization 已被证明能有效降低网络训练难度，但它依赖 batch size，因此在一些由于硬件限制而必须使用较小输入 batch size 的复杂网络中不太适用。

为缓解这个问题，人们提出了几种 batch normalization 的替代方法；其中一种是 Layer Normalization [2]。Layer Normalization 不在 batch 维度上归一化，而是在特征维度上归一化。换句话说，使用 Layer Normalization 时，每个单独数据点对应的特征向量会根据该特征向量内部所有项的统计量进行归一化。

[2] [Ba, Jimmy Lei, Jamie Ryan Kiros, and Geoffrey E. Hinton. "Layer Normalization." stat 1050 (2016): 21.](https://arxiv.org/pdf/1607.06450.pdf)

## 内联问题 3：
下列哪些数据预处理步骤类似于 batch normalization，哪些类似于 layer normalization？

1. 对数据集中的每张图像进行缩放，使图像内每一行像素的 RGB 通道和为 1。
2. 对数据集中的每张图像进行缩放，使图像内所有像素的 RGB 通道总和为 1。
3. 从数据集中的每张图像中减去数据集的均值图像。
4. 根据给定阈值，把所有 RGB 值设置为 0 或 1。

## 回答：
[在此填写]

# Layer Normalization：实现

现在你将实现 layer normalization。这一步应该相对直接，因为从概念上看，它的实现几乎与 batch normalization 相同。一个重要区别是：对于 layer normalization，我们不维护移动均值/方差；测试阶段和训练阶段相同，均值和方差都直接按每个数据点计算。

你需要完成：

* 在 `cs231n/layers.py` 中，在 `layernorm_forward` 函数里实现 layer normalization 的前向传播。

运行下面的单元检查结果。
* 在 `cs231n/layers.py` 中，在 `layernorm_backward` 函数里实现 layer normalization 的反向传播。

运行第二个单元检查结果。
* 修改 `cs231n/classifiers/fc_net.py`，为 `FullyConnectedNet` 添加 layer normalization。当构造函数中的 `normalization` 标志设置为 `"layernorm"` 时，你应该在每个 ReLU 非线性之前插入一个 layer normalization 层。

运行第三个单元，在 layer normalization 上执行 batch size 实验。

In [ ]:
# Check 训练时 前向传播 by checking 均值 并 方差
# of 特征 both before 并 after 层 归一化.

# Simulate 前向传播 用于 a two-层 network.
np.random.seed(231)
N, D1, D2, D3 =4, 50, 60, 3
X = np.random.randn(N, D1)
W1 = np.random.randn(D1, D2)
W2 = np.random.randn(D2, D3)
a = np.maximum(0, X.dot(W1)).dot(W2)

print('Before layer normalization:')
print_mean_std(a,axis=1)

gamma = np.ones(D3)
beta = np.zeros(D3)

# Means 应为 接近 zero 并 标准差 接近 one.
print('After layer normalization (gamma=1, beta=0)')
a_norm, _ = layernorm_forward(a, gamma, beta, {'mode': 'train'})
print_mean_std(a_norm,axis=1)

gamma = np.asarray([3.0,3.0,3.0])
beta = np.asarray([5.0,5.0,5.0])

# Now 均值 应为 接近 beta 并 标准差 接近 gamma.
print('After layer normalization (gamma=', gamma, ', beta=', beta, ')')
a_norm, _ = layernorm_forward(a, gamma, beta, {'mode': 'train'})
print_mean_std(a_norm,axis=1)

In [ ]:
# Gradient check batchnorm 反向传播.
np.random.seed(231)
N, D = 4, 5
x = 5 * np.random.randn(N, D) + 12
gamma = np.random.randn(D)
beta = np.random.randn(D)
dout = np.random.randn(N, D)

ln_param = {}
fx = lambda x: layernorm_forward(x, gamma, beta, ln_param)[0]
fg = lambda a: layernorm_forward(x, a, beta, ln_param)[0]
fb = lambda b: layernorm_forward(x, gamma, b, ln_param)[0]

dx_num = eval_numerical_gradient_array(fx, x, dout)
da_num = eval_numerical_gradient_array(fg, gamma.copy(), dout)
db_num = eval_numerical_gradient_array(fb, beta.copy(), dout)

_, cache = layernorm_forward(x, gamma, beta, ln_param)
dx, dgamma, dbeta = layernorm_backward(dout, cache)

# 你应该 expect 到 see 相对误差s between 1e-12 并 1e-8.
print('dx error: ', rel_error(dx_num, dx))
print('dgamma error: ', rel_error(da_num, dgamma))
print('dbeta error: ', rel_error(db_num, dbeta))

# Layer Normalization 与 Batch Size

现在我们用 layer normalization 代替 batch normalization，重新运行前面的 batch size 实验。相比前一个实验，你应该看到 batch size 对训练历史的影响明显更小。

In [ ]:
ln_solvers_bsize, solver_bsize, batch_sizes = run_batchsize_experiments('layernorm')

plt.subplot(2, 1, 1)
plot_training_history('Training accuracy (Layer Normalization)','Epoch', solver_bsize, ln_solvers_bsize,\
                      lambda x: x.train_acc_history, bl_marker='-^', bn_marker='-o', labels=batch_sizes)
plt.subplot(2, 1, 2)
plot_training_history('Validation accuracy (Layer Normalization)','Epoch', solver_bsize, ln_solvers_bsize,\
                      lambda x: x.val_acc_history, bl_marker='-^', bn_marker='-o', labels=batch_sizes)

plt.gcf().set_size_inches(15, 10)
plt.show()

## 内联问题 4：
Layer normalization 在什么情况下可能效果不好？为什么？

1. 在非常深的网络中使用。
2. 特征维度非常小。
3. 正则化项很大。

## 回答：
[在此填写]